In [1]:
import pandas as pd
import joblib
import torch
import torch.nn as nn

In [ ]:
# --> [female,29,188.0,85.0,16.0,102.0,40.4]

In [2]:
df = pd.DataFrame({
    "Sex":"female",
    "Age":29,
    "Height":188,
    "Weight":85,
    "Duration":16,
    "Heart_Rate":102,
    "Body_Temp":40.4
},index=[0])

In [3]:
df

,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp
0,female,29,188,85,16,102,40.4


In [4]:
ohe = joblib.load(filename="OneHotEncoder.pkl")
x_train_x_test_scaler = joblib.load(filename="x_train_x_test_scaler.pkl")
y_train_y_test_scaler = joblib.load(filename="y_train_y_test_scaler.pkl")

In [5]:
encoded = ohe.transform(df[["Sex"]])

In [6]:
encoded_col_name = ohe.get_feature_names_out()
encoded_col_name

array(['Sex_male'], dtype=object)

In [7]:
encoded_df = pd.DataFrame(encoded,columns=encoded_col_name)

In [8]:
encoded_df

,Sex_male
0,0.0


In [9]:
final_df = pd.concat(
    [df.drop("Sex",axis=1),encoded_df],axis=1
)

In [10]:
final_df

,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Sex_male
0,29,188,85,16,102,40.4,0.0


In [11]:
x = final_df.values

In [12]:
x_train = x_train_x_test_scaler.transform(x)

In [24]:
x_train = torch.tensor(x_train,dtype=torch.float32)

In [13]:
model = nn.Sequential(
    nn.Linear(7,128),
    nn.LayerNorm(128),
    nn.Mish(),
    nn.Dropout(0.2),

    nn.Linear(128,64),
    nn.LayerNorm(64),
    nn.Mish(),
    nn.Dropout(0.2),

    nn.Linear(64,32),
    nn.LayerNorm(32),
    nn.Mish(),
    nn.Dropout(0.3),

    nn.Linear(32,1)
)

In [14]:
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

In [15]:
checkpoint_model = torch.load(
    r"models/checkpoint_249.pth"
)

In [16]:
loss = checkpoint_model["Loss"]

In [18]:
loss

0.01573764905333519

In [17]:
epoch = checkpoint_model["Epochs"]

In [19]:
optimizer.load_state_dict(checkpoint_model["Optimizer"])

In [20]:
model.eval()

Sequential(
  (0): Linear(in_features=7, out_features=128, bias=True)
  (1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
  (2): Mish()
  (3): Dropout(p=0.2, inplace=False)
  (4): Linear(in_features=128, out_features=64, bias=True)
  (5): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
  (6): Mish()
  (7): Dropout(p=0.2, inplace=False)
  (8): Linear(in_features=64, out_features=32, bias=True)
  (9): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
  (10): Mish()
  (11): Dropout(p=0.3, inplace=False)
  (12): Linear(in_features=32, out_features=1, bias=True)
)

In [21]:
model.load_state_dict(checkpoint_model["checkpoints"])

<All keys matched successfully>

In [25]:
with torch.no_grad():
    prediction = model(x_train)

In [27]:
y_train_y_test_scaler.inverse_transform(prediction)

array([[87.89363151]])